# grid  —  Sec 5.1  9-cell codebook/snapping grid

Frozen Qwen2.5-1.5B, split-context SQuAD. Anchors are re-measured
in-run (floor = reader's own masked cache, reference = its unmasked cache).

Results are written to `results/`, figures to `figures/`.


# Grid / codebook quantization of the KV manifold

The reconstruct line hit a wall: hitting an exact continuous on-manifold **point** fails on the residual
(cos climbs to 0.85, `logit_corr` stalls at ~0.9, transfer stays at the floor). **New idea:** quantize the
valid-key manifold `K = W_K h` into a **grid of guaranteed-on-manifold points** and **snap** every shared
key to a cell. Sharing becomes discrete *selection* among pre-certified valid points instead of continuous
regression.

**3 codebook bases × 3 snap rules = 9 arms**, plus `freeform`/`bycon` baselines.

| base | what the grid points are |
|---|---|
| `bank` | harvested **real** reader keys/values (every point is one the reader produced) |
| `vq` | learned VQ codebook of hidden `h`; key = `W_K·entry` (frozen) |
| `pca` | analytic lattice in the top-R PCA axes of real keys |

| snap | rule |
|---|---|
| `logit` | nearest in the reader's **attention-logit** metric (`q·k` through frozen `W_Q`, RoPE) |
| `raw` | nearest in raw key cosine |
| `topk` | **soft** — convex mix of the *k* nearest certified-on-manifold points |

Probe says: a 9k real-key bank covers held-out keys only to cos ~0.8 → hard-snap ≈ bycon; the bet is on
`topk` (a convex mix of real keys can interpolate toward the target while staying on-manifold — this is
attend-don't-reconstruct in codebook form).

In [ ]:
import os
# Write next to this notebook's parent (the repo root), not the CWD, so
# results land in the same results/ that make_figs.py and verify_results.py read.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
import os
CACHE = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
os.environ["HF_HOME"]=CACHE; os.environ["HF_DATASETS_CACHE"]=CACHE
os.environ["TOKENIZERS_PARALLELISM"]="false"
import json, numpy as np, re, itertools
import torch, torch.nn as nn, torch.nn.functional as F
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib","inline")
except NameError:
    pass  # running as a plain script, not under IPython
assert torch.cuda.is_available(), (
    "This experiment needs a CUDA GPU. The reader runs in fp32 (deliberately: the\n"
    "cosine and attention-logit measurements are sensitive to reduced precision),\n"
    "so expect ~8 GB and more for the larger notebooks.")
DEVICE="cuda"; SEED=0; torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:",torch.cuda.get_device_name(0),flush=True)
BASE="Qwen/Qwen2.5-1.5B"
CFG=dict(
  bank_ctx=120,          # SQuAD contexts harvested to build the real-key bank
  bank_cap=6000,         # max codebook entries per layer (per K, per V) for bank/vq target size
  vq_size=2048, vq_epochs=6, vq_lr=3e-3, vq_enc_hidden=1024,  # VQ codebook of hidden h
  pca_rank=16, pca_levels=7,   # analytic lattice: top-R PCA axes, L levels/axis
  topk=8,                # soft rule: mix of k nearest on-manifold points
  n_train=200, n_eval=200, max_ctx=256, gen_len=8,
  alphas=[0.0,0.1,0.25,0.5,0.75,1.0],
  bases=["bank","vq","pca"], snaps=["logit","raw","topk"],
  baselines=["freeform","bycon"],
)
if os.environ.get("KVL_FAST","0")=="1":
    CFG.update(bank_ctx=30,bank_cap=1500,vq_size=512,vq_epochs=2,n_train=24,n_eval=24,topk=4)
    print(">>> FAST <<<",flush=True)
# A fast run writes to a separate tag so it cannot overwrite the
# committed paper results.
_default_tag = "_gridfast" if os.environ.get("KVL_FAST","0")=="1" else "_grid"
TAG=os.environ.get("KVL_TAG", _default_tag)
print("CFG",CFG,"TAG",TAG,flush=True)

## 1. Load frozen base; grab W_Q/W_K/W_V (+biases) and RoPE (identical rig)

In [ ]:
tok=AutoTokenizer.from_pretrained(BASE,cache_dir=CACHE)
if tok.pad_token is None: tok.pad_token=tok.eos_token
base=AutoModelForCausalLM.from_pretrained(BASE,cache_dir=CACHE,torch_dtype=torch.float32).to(DEVICE).eval()
for p in base.parameters(): p.requires_grad_(False)
ids=tok("hi",return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad(): pkv=base(ids,use_cache=True).past_key_values
N_LAYER=len(pkv); _,N_KV,_,D_HEAD=pkv[0][0].shape
N_Q=base.config.num_attention_heads; GROUP=N_Q//N_KV; KV_DIM=2*N_KV*D_HEAD
KDIM=N_KV*D_HEAD                      # flattened key (and value) width
HID=base.config.hidden_size
layers=base.model.layers
WQ=[layers[l].self_attn.q_proj.weight.detach() for l in range(N_LAYER)]
BQ=[getattr(layers[l].self_attn.q_proj,'bias',None) for l in range(N_LAYER)]
WK=[layers[l].self_attn.k_proj.weight.detach() for l in range(N_LAYER)]
BK=[getattr(layers[l].self_attn.k_proj,'bias',None) for l in range(N_LAYER)]
WV=[layers[l].self_attn.v_proj.weight.detach() for l in range(N_LAYER)]
BV=[getattr(layers[l].self_attn.v_proj,'bias',None) for l in range(N_LAYER)]
print(f"L={N_LAYER} n_q={N_Q} n_kv={N_KV} group={GROUP} d_head={D_HEAD} kdim={KDIM} hid={HID}",flush=True)
_THETA=float(getattr(base.config,'rope_theta',1e6))
_INV=1.0/(_THETA**(torch.arange(0,D_HEAD,2,device=DEVICE).float()/D_HEAD))
def _cossin(T):
    pos=torch.arange(T,device=DEVICE).float(); ang=torch.outer(pos,_INV); emb=torch.cat([ang,ang],-1)
    return emb.cos()[None], emb.sin()[None]
def rope(x,T):
    cos,sin=_cossin(T); t1,t2=x[...,:D_HEAD//2],x[...,D_HEAD//2:]
    rr=torch.cat([-t2,t1],-1); return x*cos+rr*sin

## 2. Cache tooling + reserved-last-token decode (identical)

In [ ]:
def kv_stack(past):
    outs=[]
    for (k,v) in past:
        B,H,T,dh=k.shape
        outs.append(torch.cat([k.permute(0,2,1,3).reshape(B,T,H*dh),
                               v.permute(0,2,1,3).reshape(B,T,H*dh)],-1))
    return torch.stack(outs,0)
def native_ids(text,max_len):
    return tok([text],return_tensors="pt",truncation=True,max_length=max_len).input_ids.to(DEVICE)
@torch.no_grad()
def ctx_cache(text,max_len):
    ids=native_ids(text,max_len); last=ids[:,-1:]; body=ids[:,:-1]
    leg=base(body,use_cache=True).past_key_values
    if not isinstance(leg,tuple): leg=leg.to_legacy_cache() if hasattr(leg,"to_legacy_cache") else leg
    return kv_stack(leg), last, body
def slots_to_legacy(vec):
    L,B,Kk,_=vec.shape; half=KDIM; leg=[]
    for l in range(L):
        kt=vec[l,:,:,:half].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        vt=vec[l,:,:,half:].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        leg.append((kt,vt))
    return leg
def to_past(vec): return DynamicCache.from_legacy_cache(tuple(slots_to_legacy(vec)))
def _pos(s,l): return torch.arange(s,s+l,device=DEVICE).unsqueeze(0)
@torch.no_grad()
def greedy_from_prefix(prefix_vec,last_id):
    past=to_past(prefix_vec); T0=prefix_vec.shape[2]
    o=base(last_id,past_key_values=past,position_ids=_pos(T0,1),
           attention_mask=torch.ones(1,T0+1,device=DEVICE),use_cache=True)
    past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen=[cur]; L=T0+1
    for _ in range(CFG["gen_len"]-1):
        o=base(cur,past_key_values=past,position_ids=_pos(L,1),
               attention_mask=torch.ones(1,L+1,device=DEVICE),use_cache=True)
        past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen.append(cur); L+=1
    return tok.decode(torch.cat(gen,1)[0])

## 3. Split-context SQuAD (identical)

In [ ]:
def make_prompt(p,q): return f"{p}\nQuestion: {q}\nAnswer:"
def mask_answer(p,a):
    i=p.find(a); return None if i<0 else p[:i]+"____"+p[i+len(a):]
def build_qa(n,split):
    ds=datasets.load_dataset("squad",split=split,cache_dir=CACHE); items=[]
    for ex in ds:
        a=ex["answers"]["text"][0].strip()
        if not a or a not in ex["context"] or len(a.split())>4: continue
        m=mask_answer(ex["context"],a)
        if m is None: continue
        items.append(dict(q=ex["question"].strip(),full=ex["context"].strip(),masked=m.strip(),ans=a))
        if len(items)>=n: break
    return items
train_qa=build_qa(CFG["n_train"],"train"); eval_qa=build_qa(CFG["n_eval"],"validation")
print("train",len(train_qa),"eval",len(eval_qa),flush=True)
def norm(s): return re.sub(r"[^a-z0-9 ]","",s.lower()).strip()
def hit(t,g): return norm(g) in norm(t) if g else False
@torch.no_grad()
def holder_full(it): return ctx_cache(make_prompt(it["full"],it["q"]),CFG["max_ctx"])
@torch.no_grad()
def holder_masked(it): return ctx_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
@torch.no_grad()
def reader_self(it): return ctx_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
def cos_full(a,b): return F.cosine_similarity(a.reshape(-1),b.reshape(-1),dim=0).item()

## 4. Capture genuine per-layer hidden Hs[l] + queries Qs[l]

In [ ]:
@torch.no_grad()
def capture_hq(body):
    Hs={}; Qs={}; hooks=[]
    def mk(l):
        def h(mod,inp,out):
            hln=out[0]; Hs[l]=hln.detach()
            q=hln@WQ[l].T
            if BQ[l] is not None: q=q+BQ[l]
            Qs[l]=q.view(-1,N_Q,D_HEAD).permute(1,0,2)
        return h
    for l in range(N_LAYER): hooks.append(layers[l].input_layernorm.register_forward_hook(mk(l)))
    base(body,use_cache=False)
    for h in hooks: h.remove()
    return Hs,Qs

## 5. Build the codebooks (per layer, separately for K and V)

- **bank**: harvest real K,V slots over `bank_ctx` SQuAD contexts; cap to `bank_cap` per layer by random
  subsample. These are literal reader outputs (K stored pre-RoPE, exactly as sharing will inject them).
- **vq**: harvest real *hidden* `h` (the input_layernorm output) per layer; k-means init then EMA/commitment
  updates give a `vq_size` codebook of h-vectors. The grid key = `W_K·codeword` (frozen), value=`W_V·codeword`.
- **pca**: fit PCA on the bank keys (and values); keep top `pca_rank` axes; quantize each axis into
  `pca_levels` bins over its observed range → snapping projects to that subspace, rounds per-axis, lifts back.

In [ ]:
@torch.no_grad()
def harvest():
    # returns per-layer real K bank, V bank, and hidden-h bank
    Kb=[[] for _ in range(N_LAYER)]; Vb=[[] for _ in range(N_LAYER)]; Hb=[[] for _ in range(N_LAYER)]
    for it in train_qa[:CFG["bank_ctx"]]:
        vec,_,body=holder_full(it); T=vec.shape[2]
        Hs,_=capture_hq(body)
        for l in range(N_LAYER):
            Kb[l].append(vec[l,0,:,:KDIM]); Vb[l].append(vec[l,0,:,KDIM:]); Hb[l].append(Hs[l])
    def cat_cap(lst):
        out=[]
        for l in range(N_LAYER):
            X=torch.cat(lst[l],0)
            if X.shape[0]>CFG["bank_cap"]:
                idx=torch.randperm(X.shape[0],device=X.device)[:CFG["bank_cap"]]; X=X[idx]
            out.append(X.contiguous())
        return out
    return cat_cap(Kb),cat_cap(Vb),cat_cap(Hb)

Kbank,Vbank,Hbank=harvest()
print("bank per-layer sizes K",[Kbank[l].shape[0] for l in range(0,N_LAYER,6)],flush=True)

# ---- VQ codebook of hidden h (k-means init + EMA), per layer ----
@torch.no_grad()
def kmeans(X,ncb,iters=8):
    n=X.shape[0]; ncb=min(ncb,n)
    c=X[torch.randperm(n,device=X.device)[:ncb]].clone()
    for _ in range(iters):
        d=torch.cdist(X,c); a=d.argmin(1)
        for k in range(ncb):
            m=a==k
            if m.any(): c[k]=X[m].mean(0)
    return c
VQh=[kmeans(Hbank[l],CFG["vq_size"]) for l in range(N_LAYER)]     # (vq_size, HID)
print("VQ codebooks built, size",VQh[0].shape,flush=True)
# grid K/V for a codeword = frozen projection of the codeword hidden
def kv_of_h(h,l):
    K=h@WK[l].T + (BK[l] if BK[l] is not None else 0.0)
    V=h@WV[l].T + (BV[l] if BV[l] is not None else 0.0)
    return K,V
VQk=[]; VQv=[]
for l in range(N_LAYER):
    K,V=kv_of_h(VQh[l],l); VQk.append(K); VQv.append(V)

# ---- PCA lattice per layer (fit on bank K and V separately) ----
@torch.no_grad()
def pca_fit(X,r):
    mu=X.mean(0); Xc=X-mu; U,S,Vh=torch.linalg.svd(Xc,full_matrices=False)
    comp=Vh[:r]                       # (r, D)
    proj=Xc@comp.T                    # (n, r)
    lo=proj.min(0).values; hi=proj.max(0).values
    return dict(mu=mu,comp=comp,lo=lo,hi=hi)
PCk=[pca_fit(Kbank[l],CFG["pca_rank"]) for l in range(N_LAYER)]
PCv=[pca_fit(Vbank[l],CFG["pca_rank"]) for l in range(N_LAYER)]
def pca_snap(x,P,levels):
    # x (T,D) -> snap to nearest per-axis lattice point in the top-r PCA subspace, lift back to D
    z=(x-P["mu"])@P["comp"].T                       # (T,r)
    step=(P["hi"]-P["lo"])/(levels-1)+1e-9
    zq=torch.round((z-P["lo"])/step)*step+P["lo"]
    zq=torch.clamp(zq,P["lo"],P["hi"])
    return zq@P["comp"]+P["mu"]                      # (T,D)
print("PCA lattices fit (rank",CFG["pca_rank"],"levels",CFG["pca_levels"],")",flush=True)

## 6. Snap operators — map a holder KV slot to grid points, per (layer, token)

`raw` and `topk` operate directly on the holder's stored key/value (`bank`/`pca`/`vq` differ only in the
codebook they select from). `logit` scores candidates by how well `q·k` matches through the reader's frozen
`W_Q` + RoPE — but scoring every token against a 6k bank in the logit metric is O(T·N·heads); we restrict the
logit-metric search to the `logit_cand` nearest raw candidates (a re-rank), which is exact for the top region
that matters and keeps it tractable.

In [ ]:
LOGIT_CAND=32
@torch.no_grad()
def snap_layer(l, Kslot, Vslot, base_name, snap, Q=None):
    # Kslot,Vslot: (T,KDIM). Returns snapped (T,KDIM),(T,KDIM).
    # PCA is a per-token analytic projection (one candidate/token) -> raw/logit/topk collapse to the lattice snap.
    T=Kslot.shape[0]
    if base_name=="pca":
        return pca_snap(Kslot,PCk[l],CFG["pca_levels"]), pca_snap(Vslot,PCv[l],CFG["pca_levels"])
    CK,CV=(Kbank[l],Vbank[l]) if base_name=="bank" else (VQk[l],VQv[l])   # (N,KDIM)
    Kn=Kslot/(Kslot.norm(dim=1,keepdim=True)+1e-9); Cn=CK/(CK.norm(dim=1,keepdim=True)+1e-9)
    sim=Kn@Cn.T                                        # (T,N) raw cosine
    if snap=="raw":
        j=sim.argmax(1); return CK[j],CV[j]
    if snap=="topk":
        k=min(CFG["topk"],CK.shape[0]); tv,ti=sim.topk(k,dim=1)     # (T,k)
        w=torch.softmax(tv/0.05,dim=1)                              # sharp convex weights over on-manifold pts
        Ksn=torch.einsum('tk,tkd->td',w,CK[ti]); Vsn=torch.einsum('tk,tkd->td',w,CV[ti])
        return Ksn,Vsn
    if snap=="logit":
        # Re-rank the LOGIT_CAND nearest raw candidates in the reader's ATTENTION-LOGIT metric.
        # Score a candidate key placed at position t by how well q_t . rope(k_cand,t) matches
        # q_t . rope(k_holder,t) across all query heads (self-position logit; RoPE is position-diagonal here).
        assert Q is not None
        k=min(LOGIT_CAND,CK.shape[0]); _,ti=sim.topk(k,dim=1)       # (T,k)
        qh=rope(Q[l],T)                                            # (heads_q,T,dh)
        qd=qh.permute(1,0,2)                                       # (T,heads_q,dh)
        # holder target: rope its own key at each position, self-position logit per head
        kt=rope(Kslot.reshape(T,N_KV,D_HEAD).permute(1,0,2),T)     # (n_kv,T,dh)
        kt_e=kt.repeat_interleave(GROUP,0).permute(1,0,2)          # (T,heads_q,dh)
        tgt=(qd*kt_e).sum(-1)                                      # (T,heads_q)
        # candidates: (T,k,KDIM) -> per-head, rope at position t
        cand=CK[ti].reshape(T,k,N_KV,D_HEAD)                       # (T,k,n_kv,dh)
        cos,sin=_cossin(T)                                        # (1,T,dh)
        c1,c2=cand[...,:D_HEAD//2],cand[...,D_HEAD//2:]
        rr=torch.cat([-c2,c1],-1)
        cand_r=cand*cos[0].view(T,1,1,D_HEAD)+rr*sin[0].view(T,1,1,D_HEAD)   # (T,k,n_kv,dh)
        cand_e=cand_r.repeat_interleave(GROUP,dim=2)              # (T,k,heads_q,dh)
        lc=torch.einsum('thd,tkhd->tkh', qd, cand_e)             # (T,k,heads_q)
        err=((lc-tgt[:,None,:])**2).mean(-1)                      # (T,k)
        j=err.argmin(1)                                          # (T,)
        sel=ti[torch.arange(T,device=DEVICE),j]
        return CK[sel],CV[sel]
    raise ValueError(snap)

## 7. Baseline decoders (freeform / bycon) reused for calibration

In [ ]:
class MLP(nn.Module):
    def __init__(s,di,do,h):
        super().__init__(); s.net=nn.Sequential(nn.Linear(di,h),nn.GELU(),nn.Linear(h,h),nn.GELU(),nn.Linear(h,do))
    def forward(s,x): return s.net(x)
def _kv_from_hidden(hsyn):
    outs=[]
    for l in range(N_LAYER):
        K=hsyn[l]@WK[l].T + (BK[l] if BK[l] is not None else 0.0)
        V=hsyn[l]@WV[l].T + (BV[l] if BV[l] is not None else 0.0)
        outs.append(torch.cat([K,V],-1))
    return torch.stack(outs,0)
class Freeform(nn.Module):     # latent->KV directly
    def __init__(s):
        super().__init__(); s.E=MLP(KV_DIM,512,1024)
        s.dec=nn.ModuleList([MLP(512,KV_DIM,1024) for _ in range(N_LAYER)])
    def forward(s,kv):
        z=s.E(kv); return torch.stack([s.dec[l](z[l]) for l in range(N_LAYER)],0)
class ByCon(nn.Module):        # latent->h->frozen W_K/W_V
    def __init__(s):
        super().__init__(); s.E=MLP(KV_DIM,512,1024); s.trunk=MLP(512,1024,1024)
        s.head=nn.ModuleList([nn.Linear(1024,HID) for _ in range(N_LAYER)])
        s.hln=nn.ModuleList([nn.LayerNorm(HID) for _ in range(N_LAYER)])
    def hidden(s,kv):
        g=s.trunk(s.E(kv)); return torch.stack([s.hln[l](s.head[l](g[l])) for l in range(N_LAYER)],0)
    def forward(s,kv): return _kv_from_hidden(s.hidden(kv))

## 8. Apply a grid arm to a full cache

`grid_apply(base_name, snap, vec, Q)` snaps every layer's K,V slot to the codebook and returns the modified
cache. Training-free for `bank`/`pca`; `vq` uses the harvested VQ codebook (also selection-only — the encoder
that would *predict* an index is unnecessary here because we snap the holder's real KV, which is available at
share time). Baselines call their trained decoder.

In [ ]:
@torch.no_grad()
def grid_apply(base_name,snap,vec,Q):
    L,B,T,_=vec.shape; out=vec.clone()
    for l in range(N_LAYER):
        Ksn,Vsn=snap_layer(l, vec[l,0,:,:KDIM], vec[l,0,:,KDIM:], base_name, snap, Q)
        out[l,0,:,:KDIM]=Ksn; out[l,0,:,KDIM:]=Vsn
    return out
def apply_arm(arm,vec,Q,models):
    if arm=="freeform": return models["freeform"](vec)
    if arm=="bycon":    return models["bycon"](vec)
    base_name,snap=arm.split(":"); return grid_apply(base_name,snap,vec,Q)

## 9. Train the two baselines (grid arms need no training)

In [ ]:
def train_baseline(kind,data,epochs=4,lr=3e-4):
    M=(Freeform() if kind=="freeform" else ByCon()).to(DEVICE)
    opt=torch.optim.Adam(M.parameters(),lr=lr)
    for ep in range(epochs):
        idx=torch.randperm(len(data)).tolist(); tot=0.0; n=0
        for j in idx:
            it=data[j]; vec,_,body=holder_full(it); T=vec.shape[2]
            _,Q=capture_hq(body)
            kvh=M(vec)
            # manifold logit loss (same as bycon experiment) — light, just to calibrate
            half=KDIM; scale=1.0/(D_HEAD**0.5); mask=torch.triu(torch.ones(T,T,device=DEVICE),1).bool()
            loss=0.0
            for l in range(N_LAYER):
                k =vec [l,0,:,:half].reshape(T,N_KV,D_HEAD).permute(1,0,2)
                kh=kvh[l,0,:,:half].reshape(T,N_KV,D_HEAD).permute(1,0,2)
                qr=rope(Q[l],T); kr=rope(k,T).repeat_interleave(GROUP,0); krh=rope(kh,T).repeat_interleave(GROUP,0)
                Lt=(torch.einsum('htd,hsd->hts',qr,kr)*scale).masked_fill(mask,float('-inf'))
                Lh=(torch.einsum('htd,hsd->hts',qr,krh)*scale).masked_fill(mask,float('-inf'))
                vld=~mask; loss=loss+((Lh[:,vld]-Lt[:,vld])**2).mean()/((Lt[:,vld]**2).mean()+1e-6)
            loss=loss/N_LAYER
            if not torch.isfinite(loss): continue
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(M.parameters(),1.0); opt.step()
            tot+=float(loss); n+=1
        print(f"  [{kind}] ep{ep+1} loss={tot/max(1,n):.4f}",flush=True)
    return M
MODELS={}
if os.environ.get("KVL_FAST","0")=="1": EP=2
else: EP=4
for kind in CFG["baselines"]:
    print(f"train baseline {kind}",flush=True); MODELS[kind]=train_baseline(kind,train_qa,epochs=EP)

## 10. Eval harness (identical: per-layer manifold + transfer + a-sweep)

In [ ]:
@torch.no_grad()
def per_layer_manifold(arm,items,models):
    keys=["cos","logit_corr"]; acc=[{k:[] for k in keys} for _ in range(N_LAYER)]
    scale=1.0/(D_HEAD**0.5); half=KDIM
    for it in items[:min(len(items),60)]:
        vec,_,body=holder_full(it); T=vec.shape[2]; _,Q=capture_hq(body)
        kvh=apply_arm(arm,vec,Q,models)
        mask=torch.triu(torch.ones(T,T,device=DEVICE),1).bool(); vld=~mask
        for l in range(N_LAYER):
            k =vec [l,0,:,:half].reshape(T,N_KV,D_HEAD).permute(1,0,2)
            kh=kvh[l,0,:,:half].reshape(T,N_KV,D_HEAD).permute(1,0,2)
            acc[l]["cos"].append(F.cosine_similarity(kh.reshape(-1),k.reshape(-1),0).item())
            qr=rope(Q[l],T); kr=rope(k,T).repeat_interleave(GROUP,0); krh=rope(kh,T).repeat_interleave(GROUP,0)
            Lt=(torch.einsum('htd,hsd->hts',qr,kr)*scale).masked_fill(mask,float('-inf'))
            Lh=(torch.einsum('htd,hsd->hts',qr,krh)*scale).masked_fill(mask,float('-inf'))
            a=Lt[:,vld].reshape(-1); b=Lh[:,vld].reshape(-1); fin=torch.isfinite(a)&torch.isfinite(b)
            if fin.sum()>2:
                acc[l]["logit_corr"].append(torch.corrcoef(torch.stack([a[fin],b[fin]]))[0,1].item())
    return [{k:(float(np.mean(acc[l][k])) if acc[l][k] else 0.0) for k in keys} for l in range(N_LAYER)]

@torch.no_grad()
def eval_transfer(arm,items,models):
    A={"floor":0,"ceiling":0,"esrd_shared":0,"esrd_ctrl":0,"esrd_self":0}; n=0
    for it in items:
        g=it["ans"]
        v,l,_=reader_self(it);   A["floor"]  +=hit(greedy_from_prefix(v,l),g)
        v,l,b=holder_full(it);   _,Q=capture_hq(b); A["ceiling"]+=hit(greedy_from_prefix(v,l),g)
        v,l,b=holder_full(it);   _,Q=capture_hq(b); A["esrd_shared"]+=hit(greedy_from_prefix(apply_arm(arm,v,Q,models),l),g)
        v,l,b=holder_masked(it); _,Q=capture_hq(b); A["esrd_ctrl"]  +=hit(greedy_from_prefix(apply_arm(arm,v,Q,models),l),g)
        v,l,b=reader_self(it);   _,Q=capture_hq(b); A["esrd_self"]  +=hit(greedy_from_prefix(apply_arm(arm,v,Q,models),l),g)
        n+=1
    return {k:v/max(1,n) for k,v in A.items()}

@torch.no_grad()
def a_sweep(arm,items,models):
    out={f"a{a}":0 for a in CFG["alphas"]}; n=0
    for it in items:
        g=it["ans"]; own,last,_=reader_self(it); hk,_,b=holder_full(it); _,Q=capture_hq(b)
        rec=apply_arm(arm,hk,Q,models); S=min(own.shape[2],rec.shape[2])
        for a in CFG["alphas"]:
            c=own.clone(); c[:,:,:S,:]=(1-a)*own[:,:,:S,:]+a*rec[:,:,:S,:]
            out[f"a{a}"]+=hit(greedy_from_prefix(c,last),g)
        n+=1
    return {k:v/max(1,n) for k,v in out.items()}

## 11. Run all arms (2 baselines + 9 grid cells) + save + verdict

In [ ]:
ARMS=list(CFG["baselines"])+[f"{b}:{s}" for b in CFG["bases"] for s in CFG["snaps"]]
print("ARMS:",ARMS,flush=True)
RES={"config":CFG,"base":BASE,"arms":{}}
RESPATH=f"results/grid{TAG}.json"
for arm in ARMS:
    print(f"\n########## ARM {arm} ##########",flush=True)
    tr=eval_transfer(arm,eval_qa,MODELS)
    pl=per_layer_manifold(arm,eval_qa,MODELS)
    sw=a_sweep(arm,eval_qa,MODELS)
    mc=float(np.mean([d["cos"] for d in pl])); mlc=float(np.mean([d["logit_corr"] for d in pl]))
    mnlc=float(np.min([d["logit_corr"] for d in pl])); nov=int(sum(d["logit_corr"]>0.99 for d in pl))
    RES["arms"][arm]=dict(transfer=tr,per_layer=pl,sweep_acc=sw,
                          mean_cos=mc,mean_logit_corr=mlc,min_logit_corr=mnlc,n_over_wall=nov)
    print(f"[{arm}] floor={tr['floor']:.3f} ceil={tr['ceiling']:.3f} shared={tr['esrd_shared']:.3f} "
          f"ctrl={tr['esrd_ctrl']:.3f} self={tr['esrd_self']:.3f}",flush=True)
    print(f"[{arm}] cos={mc:.3f} lc mean={mlc:.3f} min={mnlc:.3f} wall={nov}/{N_LAYER}",flush=True)
    print(f"[{arm}] a-sweep "+" ".join(f"a{a}={sw[f'a{a}']:.3f}" for a in CFG["alphas"]),flush=True)
    json.dump(RES,open(RESPATH,"w"),indent=1)

print("\n=== VERDICT: does snapping to on-manifold grid points transfer? ===",flush=True)
fl=RES["arms"][ARMS[0]]["transfer"]["floor"]
for arm in ARMS:
    a=RES["arms"][arm]; t=a["transfer"]
    flag=" <-- BEATS FLOOR" if t["esrd_shared"]>t["floor"]+1e-9 else ""
    print(f"  {arm:14s}: cos {a['mean_cos']:.3f} lc {a['mean_logit_corr']:.3f} wall {a['n_over_wall']:2d}/{N_LAYER} "
          f"shared {t['esrd_shared']:.3f} ctrl {t['esrd_ctrl']:.3f} (floor {t['floor']:.3f} ceil {t['ceiling']:.3f}){flag}",flush=True)
json.dump(RES,open(RESPATH,"w"),indent=1); print("saved",RESPATH,flush=True)

## 12. Plots — the 3x3 grid heatmap + a-sweeps

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.4))
# heatmap of esrd_shared over the 3x3 grid
M=np.zeros((len(CFG["bases"]),len(CFG["snaps"])))
for i,b in enumerate(CFG["bases"]):
    for j,s in enumerate(CFG["snaps"]):
        M[i,j]=RES["arms"][f"{b}:{s}"]["transfer"]["esrd_shared"]
im=ax[0].imshow(M,cmap="viridis",vmin=0,vmax=max(0.22,M.max()))
ax[0].set_xticks(range(len(CFG["snaps"]))); ax[0].set_xticklabels(CFG["snaps"])
ax[0].set_yticks(range(len(CFG["bases"]))); ax[0].set_yticklabels(CFG["bases"])
ax[0].set_title(f"esrd_shared (floor={fl:.2f})"); plt.colorbar(im,ax=ax[0],fraction=0.046)
for i in range(M.shape[0]):
    for j in range(M.shape[1]): ax[0].text(j,i,f"{M[i,j]:.2f}",ha="center",va="center",color="w",fontsize=10)
# per-layer logit_corr for the strongest few arms
show=CFG["baselines"]+["bank:raw","bank:topk","vq:topk","pca:raw"]
for arm in show:
    if arm in RES["arms"]:
        ax[1].plot(range(N_LAYER),[d["logit_corr"] for d in RES["arms"][arm]["per_layer"]],'-o',ms=2.5,label=arm)
ax[1].axhline(0.99,ls='--',color='#888'); ax[1].set_ylim(-0.1,1.02); ax[1].set_title("per-layer logit_corr")
ax[1].legend(fontsize=6.5); ax[1].set_xlabel("layer")
# a-sweeps
for arm in show:
    if arm in RES["arms"]:
        xs=CFG["alphas"]; ax[2].plot(xs,[RES["arms"][arm]["sweep_acc"][f"a{x}"] for x in xs],'-o',label=arm)
ax[2].axhline(fl,ls='--',color='#e67e22'); ax[2].set_title("a-sweep (0=own,1=snapped)"); ax[2].legend(fontsize=6.5)
for a_ in ax: a_.grid(ls=':',alpha=.4)
fig.tight_layout(); fig.savefig("figures/grid_curves.png",dpi=150); plt.show()